# FuncCraft quick start

This notebook shows how to build one concrete `BenchmarkFunction`, export and
reload its reproducibility spec, generate a `BenchmarkSuite` from YAML, export
the generated suite manifest, and minimize a function with MinionPy.

The examples below use the compiled FuncCraft interface directly. Specs are
plain typed objects on the C++ side, so they can be exported to YAML once the
runtime pieces such as transform matrices have been materialized.

In [ ]:
import sys
from pathlib import Path

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import numpy as np
import textwrap

sys.path.append(str(Path('..').resolve()))

from funccraft import (
    BasicF,
    BasicFunctionId,
    BenchmarkFunction,
    BenchmarkSuite,
    load_suite_spec_yaml,
    make_benchmark_function_from_yaml,
    make_component,
    make_composition,
    make_composition_choice,
    make_coordinate_transform,
    make_coordinate_transform_choice,
    make_domain,
    make_function_spec,
    make_suite_spec,
    make_value_transform,
    make_value_transform_choice,
    spec_to_dict,
)

def zeros(d):
    return [0.0] * d

## Construction framework

FuncCraft builds benchmark functions by composing smaller ingredients. The
general form is

```text
f(x) = psi(phi_1(g_1(T_1(x))), ..., phi_m(g_m(T_m(x))))
```

The main terms used in FuncCraft are:

- **Base function** `g_i`: a primitive benchmark landscape such as `Sphere`,
  `Rastrigin`, or `Rosenbrock`.
- **Coordinate transform** `T_i`: maps the generated/search-space point `x`
  into the coordinate system of a base function. It controls where the
  component optimum is assigned and can also rotate, scale, or restrict the
  component to a coordinate block. `assigned_xopt` is the intended optimizer
  in the generated coordinates. `base_xopt` is obtained from the selected base
  function and is the optimizer in primitive coordinates.
- **Value transform** `phi_i`: reshapes the scalar value returned by a base
  function, for example with a power, oscillatory, or cosine-zero transform.
- **Component**: one base function together with one coordinate transform,
  one value transform, and a component value offset `f_bias`. In code this is
  one entry in `FunctionSpec.components`.
- **Composition function** `psi`: combines component values into the final
  benchmark value. `CompositionKind.None` is used for one component. The CPM
  families are weighted sum, power mean, and level well. The DPM families are
  softmax and background softmax.
- **Assigned optimum**: the generated function stores the controlled optimum
  location and value in `assigned_xopt` and `assigned_fopt`.
- **Function spec**: the typed record that describes one concrete benchmark
  function. A `BenchmarkFunction` is the executable object built from this
  spec.
- **Suite spec**: a higher-level YAML/typed record that describes how many
  functions to generate and which choices of base functions, transforms, and
  compositions are allowed.

## Available base functions

Use the string names below in a component spec's `base_function` field, or
use the matching `BasicFunctionId.<name>` enum when constructing `BasicF`
directly:


| ID | Name | Modality | Conditioning | Properties |
| ---: | --- | --- | --- | --- |
| 0 | `Sphere` | Unimodal | Well-conditioned | Separable, convex. |
| 1 | `Ellipsoidal` | Unimodal | Ill-conditioned | Separable with axis scaling. |
| 2 | `SumDifferentPowers` | Unimodal | Heterogeneous | Separable, different coordinate exponents. |
| 3 | `BuecheRastrigin` | Multimodal | Moderate | Separable, asymmetric, highly periodic. |
| 4 | `LinearSlope` | Unimodal | Well-conditioned | Separable, nonsmooth at optimum. |
| 5 | `AttractiveSector` | Unimodal | Strongly anisotropic | Nonseparable when externally rotated. |
| 6 | `StepEllipsoidal` | Unimodal | Ill-conditioned | Nonsmooth step landscape. |
| 7 | `StepRastrigin` | Multimodal | Moderate | Separable, discontinuous step Rastrigin. |
| 8 | `Rosenbrock` | Unimodal | Valley-conditioned | Nonseparable, narrow curved valley. |
| 9 | `Ackley` | Multimodal | Moderate | Nonseparable, many local minima. |
| 10 | `Rastrigin` | Multimodal | Moderate | Separable, highly periodic. |
| 11 | `Griewank` | Multimodal | Moderate | Nonseparable oscillatory product term. |
| 12 | `Schwefel` | Multimodal | Deceptive | Separable, rugged. |
| 13 | `SharpRidge` | Unimodal | Ridge-conditioned | Nonseparable ridge-shaped valley. |
| 14 | `Weierstrass` | Multimodal | Rugged | Fractal ruggedness; nonseparable when rotated. |
| 15 | `SchafferF7` | Multimodal | Moderate | Nonseparable pairwise radial coupling. |
| 16 | `SchafferF7Cond1000` | Multimodal | High-conditioned | Conditioned Schaffer F7 variant. |
| 17 | `GriewankRosenbrock` | Multimodal | Valley-conditioned | Nonseparable funnel-like composition. |
| 18 | `Gallagher21` | Multimodal | Peak-conditioned | Nonseparable, few dominant peaks. |
| 19 | `Katsuura` | Multimodal | Rugged | Nonseparable, highly rugged. |
| 20 | `LunacekBiRastrigin` | Multimodal | Deceptive | Double-funnel landscape. |
| 21 | `Zakharov` | Unimodal | Coupled | Nonseparable polynomial coupling. |
| 22 | `Levy` | Multimodal | Moderate | Nonseparable periodic structure. |
| 23 | `Michalewicz` | Multimodal | Sharp minima | Nonseparable, many sharp local minima. |
| 24 | `DixonPrice` | Unimodal | Valley-conditioned | Nonseparable curved valley. |
| 25 | `BentCigar` | Unimodal | Extreme ill-conditioned | Separable with extreme axis scaling. |
| 26 | `Discus` | Unimodal | Extreme axis scaling | Separable, one dominant scaled axis. |
| 27 | `HappyCat` | Multimodal | Ridge-like | Nonseparable flat ridge structure. |
| 28 | `HGBat` | Multimodal | Ridge-like | Nonseparable flat ridge structure. |
| 29 | `HCF` | Unimodal | Moderate | Separable, exponential growth with L1 norm. |
| 30 | `GrieRosen` | Multimodal | Valley-conditioned | Griewank-Rosenbrock style coupling. |
| 31 | `SchafferF6` | Multimodal | Moderate | Nonseparable pairwise radial coupling. |
| 32 | `Step` | Unimodal | Piecewise constant | Separable step landscape. |
| 33 | `Quartic` | Unimodal | Polynomial | Separable degree-4 polynomial. |
| 34 | `Exponential` | Unimodal | Smooth | Separable smooth basin. |
| 35 | `StyblinskiTang` | Multimodal | Moderate | Separable, many local minima. |


## Creating a composed `BenchmarkFunction`

This example constructs one two-dimensional benchmark function from a user
chosen list of base functions. Edit `base_function_choices` to use any
implemented base functions from the table above. Component optimum locations
are assigned randomly inside the search box, except `origin_component_index`,
which is assigned to the origin.

The example uses the public `make_*` factory helpers from `funccraft.spec`.
Those helpers return the native C++ spec objects, so there is no separate
Python-side spec model to learn. Each component uses a rotation coordinate
transform. The transform's `assigned_xopt` is the desired component optimum in
generated coordinates; `base_xopt` is read from the primitive `BasicF` object.
The composition uses `dpm-bgsoftmax`, so component centers come from each
component's coordinate transform and component offsets come from each
component's `f_bias`.

A constructed `BenchmarkFunction` is callable and batched: pass a list of
points, not one flat vector. Useful members are `dimension`, `domain`,
`scale_factor`, `assigned_fopt`, `spec`, `evaluate(points)`, `__call__(points)`,
and `export_spec(path)`.

In [ ]:
dimension = 2
rng = np.random.default_rng(1)

base_function_choices = [
    BasicFunctionId.LunacekBiRastrigin,
    BasicFunctionId.Rosenbrock,
    BasicFunctionId.HGBat,
]
origin_component_index = len(base_function_choices) - 1

assigned_xopt = []
for index, _ in enumerate(base_function_choices):
    if index == origin_component_index:
        assigned_xopt.append(zeros(dimension))
    else:
        assigned_xopt.append(rng.uniform(-8.0, 8.0, size=dimension).tolist())

component_seeds = [17 + 6 * index for index in range(len(base_function_choices))]
component_biases = [10.0 * index for index in range(len(base_function_choices))]
component_names = [choice.name for choice in base_function_choices]

components = []
component_data = []
for choice, name, xopt, f_bias, seed in zip(
    base_function_choices,
    component_names,
    assigned_xopt,
    component_biases,
    component_seeds,
):
    base_function = BasicF(choice, dimension)
    components.append(
        make_component(
            base_function=choice,
            component_dimension=dimension,
            coordinate_transform=make_coordinate_transform(
                kind='rotation',
                dimension=dimension,
                assigned_xopt=xopt,
                base_xopt=base_function.x_opt,
                seed=seed,
            ),
            value_transform=make_value_transform('none'),
            f_bias=f_bias,
            seed=seed,
        )
    )
    component_data.append(
        {
            'name': name,
            'assigned_xopt': list(xopt),
            'base_xopt': list(base_function.x_opt),
            'f_bias': f_bias,
        }
    )

func_spec = make_function_spec(
    dimension=dimension,
    domain=make_domain(dimension, -10.0, 10.0),
    components=components,
    composition=make_composition('dpm-bgsoftmax', parameters=[0.01, 1.0, 0.01], seed=1),
    assigned_xopt=assigned_xopt[0],
    assigned_fopt=component_biases[0],
    seed=1,
    label=f"C=DPM-BGSOFTMAX;P=NONE;T=ROT;G={'+'.join(component_names)}",
)

f = BenchmarkFunction(func_spec)
print('dimension:', f.dimension)
print('label:', f.spec.label)
print('assigned_xopt:', list(f.spec.assigned_xopt))
print('assigned_fopt:', f.spec.assigned_fopt)
print('scale_factor:', f.scale_factor)
for component in component_data:
    value = f([component['assigned_xopt']])[0]
    print(f"f({component['name']} assigned_xopt): {value}")

Plot the composed function in 3D over the same `[-10, 10]^2` domain.


In [ ]:
grid_x = np.linspace(-8.0, 8.0, 120)
grid_y = np.linspace(-8.0, 8.0, 120)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]
values = np.asarray(f(grid_points), dtype=float).reshape(xx.shape)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(xx, yy, values, cmap='viridis', linewidth=0.0, antialiased=True, alpha=0.95)
ax.contour(xx, yy, values, zdir='z', offset=float(np.nanmin(values)), cmap='viridis', linewidths=0.5)

colors = plt.get_cmap('tab10')
for index, component in enumerate(component_data):
    xopt = component['assigned_xopt']
    value = f([xopt])[0]
    ax.scatter(
        [xopt[0]],
        [xopt[1]],
        [value],
        color=colors(index % 10),
        edgecolor='black',
        s=40,
        label=f"{component['name']} assigned_xopt",
    )

ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('f(x)')
ax.set_title(f"DPM-BGSoftmax composition: {' + '.join(component_names)}")
ax.view_init(elev=10, azim=-120)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

Minimize the same function with MinionPy. Install it first if the package is
not already available in the active notebook environment.


In [ ]:
# Uncomment if MinionPy is not installed in this environment.
# %pip install minionpy

import minionpy as mpy


def objective_function(x):
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        return f([x.tolist()])[0]
    return f(x.tolist())


x0 = [
    [0.0] * dimension,
]

optimizer = mpy.Minimizer(
    func=objective_function,
    x0=x0,
    bounds=[(-10, 10)] * dimension,
    algo='rcmaes',
    maxevals=10000,
    callback=None,
    seed=None,
    options=None,
)
result = optimizer.optimize()
print(result)
print(f'Solution: {result.x}')
print(f'Function value: {result.fun}')

Export the function spec to YAML and read it back. The exported file contains
the normalized runtime spec, including the generated rotation matrices,
component `base_xopt`, component `assigned_xopt`, component `f_bias`, and the
function-level `assigned_xopt`, `assigned_fopt`, and `scale_factor`.

In [ ]:
function_yaml_path = Path('manual_function.yaml')
f.export_spec(str(function_yaml_path))
reloaded_f = make_benchmark_function_from_yaml(str(function_yaml_path))
check_points = [func_spec.assigned_xopt, [0.5, -0.25], [-1.0, 1.5]]
print('wrote:', function_yaml_path.resolve())
print('before:', f(check_points))
print('after :', reloaded_f(check_points))

## Benchmark suite

Here we construct a suite spec with the same public factory helpers, generate
200 functions at dimension 2, then plot all of them in batches of 24 per PDF
page with wrapped titles.

You can also load the packaged YAML suite directly with
`load_suite_spec_yaml('../BenchmarkSuites/default_suite.yaml')`; using
`make_suite_spec` below makes the allowed choices explicit in the notebook.

In [ ]:
suite_spec = make_suite_spec(
    supported_dimensions='2',
    requested_number_of_functions=200,
    max_components=4,
    master_seed=1,
    lower_bound=-100.0,
    upper_bound=100.0,
    assigned_fopt=100.0,
    xopt_domain_shrink_factor=0.8,
    suite_label='notebook',
    coordinate_transforms=[
        make_coordinate_transform_choice('none', 0.0),
        make_coordinate_transform_choice('rotation', 0.5),
        make_coordinate_transform_choice('affine', 0.0),
        make_coordinate_transform_choice('block-rotation', 0.5),
    ],
    value_transforms=[
        make_value_transform_choice('none', 0.5),
        make_value_transform_choice('power', 0.25, [1.0, 1.0]),
        make_value_transform_choice('oscillatory', 0.25, [0.1, 1.0]),
        make_value_transform_choice('cosine-zero', 0.0, [1.0]),
    ],
    compositions=[
        make_composition_choice('cpm-wsum', 0.1),
        make_composition_choice('cpm-power-mean', 0.1, [3.0]),
        make_composition_choice('cpm-power-mean', 0.1, [0.1]),
        make_composition_choice('cpm-level-well', 0.2, [0.1, 1.0]),
        make_composition_choice('dpm-softmax', 0.25, [0.01]),
        make_composition_choice('dpm-bgsoftmax', 0.25, [0.01, 1.0, 0.01]),
    ],
)

# A plain dict view is useful for inspection in notebooks.
spec_to_dict(suite_spec)['compositions'][:2]

def wrap_title(idx, label):
    fields = {}
    for part in label.split(';'):
        if '=' in part:
            key, value = part.split('=', 1)
            fields[key.strip()] = value.strip()
    lines = [f'{idx}:']
    cpt = '; '.join(f'{key}={fields[key]}' for key in ('C', 'P', 'T') if key in fields)
    if cpt:
        lines.append(cpt)
    g_value = fields.get('G', '')
    if g_value:
        terms = g_value.split('+')
        if len(terms) <= 1:
            lines.append('G=' + g_value)
        else:
            best_split = 1
            best_score = None
            for split in range(1, len(terms)):
                left = '+'.join(terms[:split])
                right = '+'.join(terms[split:])
                score = max(len(left), len(right))
                if best_score is None or score < best_score:
                    best_score = score
                    best_split = split
            lines.append('G=' + '+'.join(terms[:best_split]))
            lines.append('+'.join(terms[best_split:]))
    return '\n'.join(lines)

In [ ]:
suite = BenchmarkSuite(suite_spec, 2)
print('size:', suite.size)
print('dimension:', suite.dimension)
print('max_number_of_functions:', suite.max_number_of_functions)
print('theoretical_max_number_of_functions:', suite.theoretical_max_number_of_functions)


Export the suite manifest to YAML. This writes the normalized suite spec and
all generated function specs, so the exact benchmark set can be reused later.


In [ ]:
suite_yaml_path = Path('generated_suite_manifest.yaml')
suite.export_manifest(str(suite_yaml_path))
reloaded_suite = BenchmarkSuite(str(suite_yaml_path), suite.dimension)
print('wrote:', suite_yaml_path.resolve())
print('reloaded size:', len(reloaded_suite))

In [ ]:
selected_indices = list(range(len(suite)))
grid_x = np.linspace(-100.0, 100.0, 50)
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

num_columns = 4
num_rows = 6
per_page = num_columns * num_rows
pdf_path = Path('benchmark_suite_100_dim2_log.pdf')

with PdfPages(pdf_path) as pdf:
    for page_start in range(0, len(selected_indices), per_page):
        page_indices = selected_indices[page_start:page_start + per_page]
        fig = plt.figure(figsize=(16, 24))
        for pos, idx in enumerate(page_indices, start=1):
            function = suite.function(idx)
            values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
            plot_values = np.log10(np.maximum(values - np.nanmin(values) + 1.0e-12, 1.0e-12))
            zmin = float(np.nanpercentile(plot_values, 5))
            ax = fig.add_subplot(num_rows, num_columns, pos, projection='3d')
            ax.plot_surface(xx, yy, plot_values, cmap='viridis', linewidth=0.3, antialiased=True)
            ax.contour(xx, yy, plot_values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
            ax.set_title(wrap_title(idx, function.spec.label), fontsize=6, pad=14)
            ax.set_xlabel('x1', fontsize=7)
            ax.set_ylabel('x2', fontsize=7)
            ax.set_zlabel('log10(f - min + eps)', fontsize=7)
            ax.tick_params(labelsize=6)
            ax.view_init(elev=18, azim=-135)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}')

In [ ]:
dpm_suite_spec = make_suite_spec(
    supported_dimensions='2',
    requested_number_of_functions=200,
    max_components=4,
    compositions=[make_composition_choice('dpm-softmax', 1.0, [0.005])],
)

dpm_suite = BenchmarkSuite(dpm_suite_spec, 2)
dpm_indices = [49, 50, 57]
dpm_labels = ['F42', 'F45', 'F48']
grid_x = np.linspace(-100.0, 100.0, 50)
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

fig = plt.figure(figsize=(18, 6))
for pos, (idx, label) in enumerate(zip(dpm_indices, dpm_labels), start=1):
    function = dpm_suite.function(idx)
    values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
    plot_values = np.log10(np.maximum(values - np.nanmin(values) + 1.0e-12, 1.0e-12))
    zmin = float(np.nanpercentile(plot_values, 5))
    ax = fig.add_subplot(1, 3, pos, projection='3d')
    ax.plot_surface(xx, yy, plot_values, cmap='viridis', linewidth=0.3, antialiased=True)
    ax.contour(xx, yy, plot_values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
    ax.set_title(wrap_title(idx, f"{label}: {function.spec.label}"), fontsize=7)
    ax.set_xlabel('x1', fontsize=7)
    ax.set_ylabel('x2', fontsize=7)
    ax.set_zlabel('log10(f - min + eps)', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.view_init(elev=10, azim=45)

plt.tight_layout()
plt.show()

In [ ]:
dpm_suite_spec = make_suite_spec(
    supported_dimensions='2',
    requested_number_of_functions=200,
    max_components=4,
    compositions=[make_composition_choice('dpm-bgsoftmax', 1.0, [0.01, 1.0, 0.01])],
)

dpm_suite = BenchmarkSuite(dpm_suite_spec, 2)
dpm_indices = [49, 50, 57]
dpm_labels = ['F42', 'F45', 'F48']
grid_x = np.linspace(-100.0, 100.0, 50)
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

fig = plt.figure(figsize=(18, 6))
for pos, (idx, label) in enumerate(zip(dpm_indices, dpm_labels), start=1):
    function = dpm_suite.function(idx)
    values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
    plot_values = np.log10(np.maximum(values - np.nanmin(values) + 1.0e-12, 1.0e-12))
    zmin = float(np.nanpercentile(plot_values, 5))
    ax = fig.add_subplot(1, 3, pos, projection='3d')
    ax.plot_surface(xx, yy, plot_values, cmap='viridis', linewidth=0.3, antialiased=True)
    ax.contour(xx, yy, plot_values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
    ax.set_title(wrap_title(idx, f"{label}: {function.spec.label}"), fontsize=7)
    ax.set_xlabel('x1', fontsize=7)
    ax.set_ylabel('x2', fontsize=7)
    ax.set_zlabel('log10(f - min + eps)', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.view_init(elev=18, azim=-135)

plt.tight_layout()
plt.show()

## Pure base functions

This page plots the raw base functions directly, without any coordinate
transform or value transform. The layout matches the benchmark-suite
page style, and paginates 24 functions per PDF page.
Each plot uses the primitive's `default_domain`.


In [ ]:
base_function_names = [
    'Sphere',
    'SumDifferentPowers',
    'Ellipsoidal',
    'BuecheRastrigin',
    'LinearSlope',
    'AttractiveSector',
    'StepEllipsoidal',
    'StepRastrigin',
    'Rosenbrock',
    'Ackley',
    'Rastrigin',
    'Griewank',
    'Schwefel',
    'SharpRidge',
    'Weierstrass',
    'SchafferF7',
    'SchafferF7Cond1000',
    'GriewankRosenbrock',
    'Gallagher21',
    'Katsuura',
    'LunacekBiRastrigin',
    'Zakharov',
    'Levy',
    'Michalewicz',
    'DixonPrice',
    'BentCigar',
    'Discus',
    'HappyCat',
    'HGBat',
    'HCF',
    'GrieRosen',
    'SchafferF6',
    'Step',
    'Quartic',
    'Exponential',
    'StyblinskiTang',
]

default_domain = BasicF(BasicFunctionId.Sphere, 2).default_domain
grid_x = np.linspace(-2, 1, 50)
grid_y = np.linspace(-2, 1, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

def wrap_title(idx, label, width=14):
    wrapped = textwrap.wrap(label, width=width, break_long_words=False, break_on_hyphens=False)
    return f"{idx}:\n" + "\n".join(wrapped)

pdf_path = Path('pure_base_functions_dim2.pdf')
num_columns = 4
num_rows = 6
per_page = num_columns * num_rows

with PdfPages(pdf_path) as pdf:
    for page_start in range(0, len(base_function_names), per_page):
        page_names = base_function_names[page_start:page_start + per_page]
        fig = plt.figure(figsize=(16, 24))
        for local_pos, base_name in enumerate(page_names, start=1):
            global_pos = page_start + local_pos - 1
            function = BasicF(getattr(BasicFunctionId, base_name), 2)
            values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
            zmin = float(np.nanpercentile(values, 5))
            zmax = float(np.nanpercentile(values, 95))
            ax = fig.add_subplot(num_rows, num_columns, local_pos, projection="3d")
            try :
                ax.plot_surface(xx, yy, values, cmap="viridis", linewidth=0.3, antialiased=True)
            except : 
                continue
            ax.contour(xx, yy, values, zdir="z", offset=zmin, cmap="viridis", linewidths=0.5)
            ax.set_title(wrap_title(global_pos, base_name), fontsize=5, pad=12)
            ax.set_xlabel("x1", fontsize=7)
            ax.set_ylabel("x2", fontsize=7)
            ax.set_zlabel("f", fontsize=7)
            ax.set_zlim(zmin, zmax)
            ax.tick_params(labelsize=6)
            ax.view_init(elev=18, azim=-135)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Saved {pdf_path}")
